# 1. Data Collection

## Purpose

This notebook collects the raw datasets used to investigate crude-oil volatility and market regimes.

The project will combine market-price data with energy fundamentals and macroeconomic indicators. Raw datasets are retained separately from processed data to preserve their original form and ensure that the analysis can be reproduced.

## Data Sources

The initial dataset consists of daily West Texas Intermediate (WTI) crude-oil spot prices. Additional energy-market and macroeconomic variables will be incorporated as the project develops.

### 1.1 WTI Crude Oil Prices

WTI is used as the primary representation of the US crude-oil market. Daily price data will provide the basis for calculating returns, historical volatility and other market-derived features used later in the analysis.

**Source:** U.S. Energy Information Administration (EIA)  
**Series:** Cushing, Oklahoma WTI Spot Price FOB  
**Unit:** US dollars per barrel  
**Frequency:** Daily

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Loading the WTI Dataset

The raw EIA dataset is loaded into a pandas DataFrame for initial inspection. At this stage, no modifications are made to the source file.

In [4]:
wti = pd.read_csv(
    "../data/raw/Cushing_OK_WTI_Spot_Price_FOB.csv",
    skiprows=4
)

In [5]:
wti.head()

,Day,Cushing OK WTI Spot Price FOB Dollars per Barrel
0,10/6/2026,96.24
1,10/5/2026,96.13
2,10/2/2026,97.89
3,10/1/2026,99.77
4,09/30/2026,97.18


In [6]:
wti.info()

<class 'pandas.DataFrame'>
RangeIndex: 9517 entries, 0 to 9516
Data columns (total 2 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Day                                                9517 non-null   str    
 1   Cushing OK WTI Spot Price FOB  Dollars per Barrel  9517 non-null   float64
dtypes: float64(1), str(1)
memory usage: 148.8 KB


In [7]:
wti.describe()

,Cushing OK WTI Spot Price FOB Dollars per Barrel
count,9517.000000
mean,50.000858
std,29.855973
min,-36.980000
25%,20.500000
50%,47.090000
75%,73.230000
max,145.310000


### Initial Inspection

The WTI dataset contains 9,517 daily observations with no missing values in either the date or price columns.

The price series is stored numerically, while the `Day` column is currently represented as text and will require conversion to a datetime format during data cleaning.

WTI prices range from -$36.98 to $145.31 per barrel. The negative minimum is not assumed to represent a data error: WTI prices briefly became negative during the extreme oil-market disruption of April 2020. This observation will therefore be retained and investigated as part of the analysis rather than automatically treated as an erroneous outlier.

The mean WTI price across the dataset is approximately $50.00 per barrel, compared with a median of $47.09.

In [8]:
wti.tail()

,Day,Cushing OK WTI Spot Price FOB Dollars per Barrel
9512,01/8/1986,25.87
9513,01/7/1986,25.85
9514,01/6/1986,26.53
9515,01/3/1986,26.00
9516,01/2/1986,25.56


The dataset covers the period from 2 January 1986 to 6 October 2026, providing more than 40 years of historical WTI price data across substantially different market conditions.

### 1.2 U.S. Commercial Crude Oil Inventories

Weekly U.S. commercial crude-oil inventory data are obtained from the U.S. Energy Information Administration (EIA).

Inventories provide a measure of the balance between crude-oil supply and demand. Changes in inventory levels may therefore contain information about market conditions that is not captured by price data alone.

Commercial inventories are used rather than the Strategic Petroleum Reserve (SPR), as the objective is to capture inventories associated more directly with normal commercial oil-market activity.

**Source:** U.S. Energy Information Administration (EIA)  
**Series:** U.S. Ending Stocks of Crude Oil, excluding SPR  
**Unit:** Thousand barrels  
**Frequency:** Weekly

In [14]:
inventory = pd.read_excel(
    "../data/raw/WCESTUS1w.xls",
    sheet_name="Data 1"
)

In [15]:
inventory.head()

,Back to Contents,Data 1: Weekly U.S. Ending Stocks excluding SPR of Crude Oil (Thousand Barrels)
0,Sourcekey,WCESTUS1
1,Date,Weekly U.S. Ending Stocks excluding SPR of Cru...
2,1982-08-20 00:00:00,338764
3,1982-08-27 00:00:00,336138
4,1982-09-24 00:00:00,335586


In [16]:
inventory.tail()

,Back to Contents,Data 1: Weekly U.S. Ending Stocks excluding SPR of Crude Oil (Thousand Barrels)
2294,2026-09-04 00:00:00,424069
2295,2026-09-11 00:00:00,423429
2296,2026-09-18 00:00:00,426398
2297,2026-09-25 00:00:00,427320
2298,2026-10-02 00:00:00,424134


In [17]:
inventory.info()

<class 'pandas.DataFrame'>
RangeIndex: 2299 entries, 0 to 2298
Data columns (total 2 columns):
 #   Column                                                                            Non-Null Count  Dtype 
---  ------                                                                            --------------  ----- 
 0   Back to Contents                                                                  2299 non-null   object
 1   Data 1: Weekly U.S. Ending Stocks excluding SPR of Crude Oil  (Thousand Barrels)  2299 non-null   object
dtypes: object(2)
memory usage: 36.1+ KB


### Initial Inspection

The EIA workbook contains a separate contents sheet and data sheet. The `Data 1` worksheet contains the weekly U.S. commercial crude-oil inventory series.

The worksheet contains 2,299 rows, with the first two rows containing metadata rather than observations. The inventory series therefore contains 2,297 weekly observations and covers the period from 20 August 1982 to 2 October 2026.

Both columns are currently represented as generic objects rather than appropriate date and numerical data types. This results from the metadata included at the beginning of the worksheet but it is going to be cleaned out.

Unlike the daily WTI price series, the inventory series is reported weekly. The series also goes back further than WTI. This difference in data frequency will need to be addressed before the datasets can be combined for modelling.

### 1.3 U.S. Crude Oil Production

Weekly U.S. crude-oil production data are obtained from the U.S. Energy Information Administration (EIA).

Domestic production represents an important component of crude-oil supply. Changes in production may therefore provide information about evolving supply conditions and their relationship with crude-oil prices, volatility and market regimes.

**Source:** U.S. Energy Information Administration (EIA)  
**Series:** U.S. Field Production of Crude Oil  
**Unit:** Thousand barrels per day  
**Frequency:** Weekly

In [18]:
pd.ExcelFile("../data/raw/WCRFPUS2w.xls").sheet_names

['Contents', 'Data 1']

In [19]:
production = pd.read_excel(
    "../data/raw/WCRFPUS2w.xls",
    sheet_name="Data 1"
)

In [20]:
production.head()

,Back to Contents,Data 1: Weekly U.S. Field Production of Crude Oil (Thousand Barrels per Day)
0,Sourcekey,WCRFPUS2
1,Date,Weekly U.S. Field Production of Crude Oil (Th...
2,1983-01-07 00:00:00,8634
3,1983-01-14 00:00:00,8634
4,1983-01-21 00:00:00,8634


In [21]:
production.tail()

,Back to Contents,Data 1: Weekly U.S. Field Production of Crude Oil (Thousand Barrels per Day)
2277,2026-09-04 00:00:00,13947
2278,2026-09-11 00:00:00,13944
2279,2026-09-18 00:00:00,13939
2280,2026-09-25 00:00:00,13955
2281,2026-10-02 00:00:00,13979


In [22]:
production.info()

<class 'pandas.DataFrame'>
RangeIndex: 2282 entries, 0 to 2281
Data columns (total 2 columns):
 #   Column                                                                         Non-Null Count  Dtype 
---  ------                                                                         --------------  ----- 
 0   Back to Contents                                                               2282 non-null   object
 1   Data 1: Weekly U.S. Field Production of Crude Oil  (Thousand Barrels per Day)  2282 non-null   object
dtypes: object(2)
memory usage: 35.8+ KB


### Initial Inspection

The EIA workbook follows the same structure as the commercial inventory dataset, containing a separate contents sheet and a `Data 1` worksheet containing the production series.

The worksheet contains 2,282 rows, with the first two rows containing metadata rather than observations. The underlying production series therefore contains approximately 2,280 weekly observations and covers the period from 7 January 1983 to 2 October 2026.

Both columns are currently represented as generic objects rather than appropriate date and numerical data types. The metadata rows will need to be removed and the data types corrected during cleaning.

The production series is reported weekly, matching the frequency of the commercial inventory series but differing from the daily WTI price series.

### 1.4 CBOE Volatility Index (VIX)

Daily observations of the CBOE Volatility Index (VIX) are obtained through the Federal Reserve Bank of St. Louis FRED database.

The VIX measures market expectations of near-term volatility derived from S&P 500 index options. It is included as an indicator of broader financial-market uncertainty and risk sentiment, allowing the analysis to investigate whether periods of heightened market stress contain information about crude-oil volatility and market regimes.

**Source:** CBOE via Federal Reserve Bank of St. Louis (FRED)  
**Series:** CBOE Volatility Index: VIX  
**Series ID:** VIXCLS  
**Unit:** Index  
**Frequency:** Daily

In [23]:
vix = pd.read_csv("../data/raw/VIXCLS.csv")

In [24]:
vix.head()

,observation_date,VIXCLS
0,2021-10-06,21.00
1,2021-10-07,19.54
2,2021-10-08,18.77
3,2021-10-11,20.00
4,2021-10-12,19.85


In [25]:
vix.tail()

,observation_date,VIXCLS
1300,2026-09-30,16.34
1301,2026-10-01,16.39
1302,2026-10-02,15.31
1303,2026-10-05,15.52
1304,2026-10-06,15.01


In [26]:
vix.info()

<class 'pandas.DataFrame'>
RangeIndex: 1305 entries, 0 to 1304
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  1305 non-null   str    
 1   VIXCLS            1288 non-null   float64
dtypes: float64(1), str(1)
memory usage: 20.5 KB


In [27]:
vix = pd.read_csv("../data/raw/VIXCLS.csv")

In [28]:
vix.head()

,observation_date,VIXCLS
0,1990-01-02,17.24
1,1990-01-03,18.19
2,1990-01-04,19.22
3,1990-01-05,20.11
4,1990-01-08,20.26


In [29]:
vix.tail()

,observation_date,VIXCLS
9586,2026-09-30,16.34
9587,2026-10-01,16.39
9588,2026-10-02,15.31
9589,2026-10-05,15.52
9590,2026-10-06,15.01


In [30]:
vix.info()

<class 'pandas.DataFrame'>
RangeIndex: 9591 entries, 0 to 9590
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  9591 non-null   str    
 1   VIXCLS            9289 non-null   float64
dtypes: float64(1), str(1)
memory usage: 150.0 KB


### Initial Inspection

The VIX dataset contains 9,591 dated rows covering the period from 2 January 1990 to 6 October 2026. Of these rows, 9,289 contain reported VIX observations, leaving 302 missing values that will require investigation during data cleaning.

The VIX values are already represented numerically, while the `observation_date` column is currently stored as text and will need to be converted to a datetime format.

The VIX series begins later than the other datasets collected so far. Consequently, including VIX in a combined modelling dataset may restrict the common analysis period to 1990 onwards.

### 1.5 Trade Weighted U.S. Dollar Index

Daily observations of the Trade Weighted U.S. Dollar Index are obtained from the Federal Reserve Bank of St. Louis FRED database.

Crude oil is internationally priced in U.S. dollars, making changes in the value of the dollar potentially relevant to oil-market behaviour. The dollar index is included to investigate whether broader currency movements contain information about crude-oil prices, volatility and market regimes.

**Source:** Board of Governors of the Federal Reserve System via FRED  
**Series:** Trade Weighted U.S. Dollar Index: Broad, Goods and Services  
**Series ID:** DTWEXBGS  
**Frequency:** Daily

In [32]:
dollar = pd.read_csv("../data/raw/DTWEXM.csv")

In [33]:
dollar.head()

,observation_date,DTWEXM
0,1973-01-02,108.2242
1,1973-01-03,108.2934
2,1973-01-04,108.2632
3,1973-01-05,108.2548
4,1973-01-08,108.3818


In [34]:
dollar.tail()

,observation_date,DTWEXM
12256,2019-12-25,NaN
12257,2019-12-26,91.9234
12258,2019-12-27,91.4593
12259,2019-12-30,91.1403
12260,2019-12-31,90.8221


In [35]:
dollar.info()

<class 'pandas.DataFrame'>
RangeIndex: 12261 entries, 0 to 12260
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  12261 non-null  str    
 1   DTWEXM            11834 non-null  float64
dtypes: float64(1), str(1)
memory usage: 191.7 KB


In [36]:
dollar_new = pd.read_csv("../data/raw/DTWEXBGS.csv")

In [37]:
dollar_new.head()

,observation_date,DTWEXBGS
0,2006-01-02,101.4155
1,2006-01-03,100.7558
2,2006-01-04,100.2288
3,2006-01-05,100.2992
4,2006-01-06,100.0241


In [38]:
dollar_new.tail()

,observation_date,DTWEXBGS
5410,2026-09-28,120.6958
5411,2026-09-29,121.0296
5412,2026-09-30,120.9693
5413,2026-10-01,121.7882
5414,2026-10-02,121.3848


In [39]:
dollar_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 5415 entries, 0 to 5414
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  5415 non-null   str    
 1   DTWEXBGS          5203 non-null   float64
dtypes: float64(1), str(1)
memory usage: 84.7 KB


### Initial Inspection

The broad trade-weighted U.S. dollar index contains 5,415 dated rows covering the period from 2 January 2006 to 2 October 2026. Of these rows, 5,203 contain reported index observations, leaving 212 missing values that will be investigated during data cleaning.

The index values are already represented numerically, while the `observation_date` column is currently stored as text and will require conversion to a datetime format.

An older Federal Reserve dollar index (`DTWEXM`) was initially considered because it provides a substantially longer historical series beginning in 1973. However, this series was discontinued in 2019. The broader `DTWEXBGS` series was selected instead because it extends through the present dataset and therefore retains important recent market periods, including the COVID-19 disruption and subsequent energy-market shocks.

The inclusion of this series will restrict the common modelling period to approximately 2006 onwards.

### 1.6 Effective Federal Funds Rate

Daily observations of the Effective Federal Funds Rate are obtained from the Federal Reserve Bank of St. Louis FRED database.

The federal funds rate is included as an indicator of U.S. monetary conditions. Changes in interest rates may influence economic activity and the U.S. dollar, potentially affecting crude-oil prices and volatility.

The inclusion of this variable will allow the project to investigate whether monetary conditions contribute useful information when identifying crude-oil market regimes.

**Source:** Federal Reserve Bank of New York via FRED  
**Series:** Effective Federal Funds Rate  
**Series ID:** DFF  
**Unit:** Percent  
**Frequency:** Daily

In [2]:
import pandas as pd

interest = pd.read_csv("../data/raw/DFF.csv")

In [3]:
interest.head()

,observation_date,DFF
0,1954-07-01,1.13
1,1954-07-02,1.25
2,1954-07-03,1.25
3,1954-07-04,1.25
4,1954-07-05,0.88


In [4]:
interest.tail()

,observation_date,DFF
26393,2026-10-04,3.88
26394,2026-10-05,3.88
26395,2026-10-06,3.88
26396,2026-10-07,3.88
26397,2026-10-08,3.88


In [5]:
interest.info()

<class 'pandas.DataFrame'>
RangeIndex: 26398 entries, 0 to 26397
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  26398 non-null  str    
 1   DFF               26398 non-null  float64
dtypes: float64(1), str(1)
memory usage: 412.6 KB


### Initial Inspection

The Effective Federal Funds Rate dataset contains 26,398 daily observations covering the period from 1 July 1954 to 8 October 2026.

Both columns are complete, with no missing values. The interest-rate values are already stored numerically, while the `observation_date` column is represented as text and will need to be converted to a datetime format during data cleaning.

Unlike the WTI, VIX and dollar datasets, the federal funds rate series includes weekends and other non-trading days. This difference in reporting frequency will need to be addressed when combining the datasets.

The series extends considerably further back than the other datasets thus its inclusion will not restrict the common modelling period.

### 1.7 Considered but Not Used: S&P 500 Index

The S&P 500 was considered as an additional variable to represent broader U.S. equity-market performance. My experience with trading tells me that equity-market movements could potentially provide useful information about investor sentiment and wider economic conditions affecting crude-oil prices.

However, the S&P 500 series available through FRED only provided around ten years of historical data, beginning in October 2016. This would reduce the available modelling period if the variable were included alongside the other datasets and make the rest of this work so far redundant.

Alternative historical data sources were explored, but a suitable longer-term dataset was not obtained during the initial data collection stage.

As such, the S&P 500 was excluded from the initial analysis to preserve the intended historical coverage and maintain a consistent approach to data collection.